# Cassette fronts: test the round 10 labels

Choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Return the final ZIP. The numerical training settings are already fixed.

This notebook tests the **r10 supervision package** after inspection of **all 126 new-series photographs**, including each full original and its annotation overlays. The **31 user-confirmed old-series review files remain byte-identical to r9**. The verification cell prints the final amendment counts, remaining cases, and actual training-input counts from the pinned manifests.

The target is **the visible cassette front and its own frontal rim**. Its two landmarks are the **actual lower corners of the front face**, ordered from left to right in the image. A face corner can lie inward from a side-face or rim silhouette. The box encloses the visible front and its frontal rim; it does not dictate point positions. A hidden or unlocatable corner stays independently missing. Camera PNGs remain unchanged.

| Run | Training supervision | Purpose |
| --- | --- | --- |
| Reference | Exact PR35 labels | Measure the accepted supervision in this runtime |
| Candidate | Eligible r10 labels; exact PR35 on held and excluded photographs | Test the reviewed amendment package |

Both arms retain **137 training photos, 13 empty training scenes, and the original 20-photo / 664-front training-validation split**. An unresolved training photograph keeps exact PR35 labels in both arms. Separately, **all 31 confirmed old-series photographs are excluded from this treatment**, also with exact PR35 in both arms. These confirmed exclusions are not unresolved holds, and neither group becomes an empty negative. Their accepted review labels remain available for inspection.

The r10 development review is separately versioned and exploratory. It does not replace training validation or the **666-front primary and 663-front historical scoring references**. The review PDF includes all 126 new-series photos; the confirmed old cohort is omitted.

Both full arms independently start from original **epoch173**: **seed 46, 30 epochs, batch 4, input size 1280, FP32, AdamW, learning rate 0.0001**. **Final epoch** selection is fixed. The four-batch resource trial checks resources and does not initialize either full run.

**PR35 remains selected.** This experiment tests the complete eligible label package; no training has already been run for r10, and this notebook makes no automatic model replacement. Completed bundles are backed up to Drive in Colab. Return unfavorable results too.


## 1. Prepare a workspace

This cell creates a workspace for the exact source and protocol pins. In Colab it offers a Drive mount to preserve completed work. The next cell verifies the original images, reviewed labels, separate held/excluded cohorts, and both fixed scoring references before installing the trainer.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REF = "bae1ed46bf3337956dbd60ac85aa5bcc759bbbcd"
PROTOCOL_PATH = "experiments/front-review-2026-11-19-r10.json"
PROTOCOL_SHA256 = "0b73d423fcaff8f7b7d72da0e6bc33acb5814f2d8435fffa0e163b3efd1bd48f"
EXPERIMENT_ID = "front-review-r10"
ROLE_RECIPES = {"reference": "reviewed-incremental-fronts-seed46-v1",
                "candidate": "reviewed-incremental-fronts-seed46-v1"}
RECIPE_SHA256 = "f18e0a813a1057e1b3ab3a44ea3ce9b2a3e7673cc2bf04ca09b394bfdd8c51b8"
RUN_BASE = Path("/content/baomai-front-review-r10")
ROLES = ("reference", "candidate")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REF), "Use the final handoff notebook with its full source commit pinned."
assert re.fullmatch(r"[0-9a-f]{64}", PROTOCOL_SHA256), "Use the final handoff notebook with its protocol hash pinned."
assert sys.version_info >= (3, 11), "Python3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

workspace_identity = {"source_revision": SOURCE_REF, "protocol_sha256": PROTOCOL_SHA256}
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    assert RUN_BASE.is_dir() and not RUN_BASE.is_symlink(), "Choose a fresh nonsymlink workspace."
    marker = RUN_BASE / "workspace-identity.json"
    assert marker.is_file() and json.loads(marker.read_text()) == workspace_identity, "Existing workspace belongs to different pins; preserve it and choose a new RUN_BASE."
else:
    RUN_BASE.mkdir(parents=True)
    (RUN_BASE / "workspace-identity.json").write_text(json.dumps(workspace_identity, indent=2) + "\n")
REPO = RUN_BASE / "dl"
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Drive is unavailable here; preserve completed bundles and the ZIP from the local workspace.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-front-review-r10-20261119")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def safe_member(name):
    p = Path(name)
    assert isinstance(name, str) and name and not p.is_absolute() and ".." not in p.parts
    assert "\\" not in name and ":" not in name and p.as_posix() == name
    return p

def artifact_inventory(path):
    path = Path(path)
    if path.is_file():
        names = [path.name]
        base = path.parent
    else:
        base = path
        receipt = json.loads((path / "training-receipt.json").read_text())
        names = ["training-receipt.json", "staging-manifest.json", receipt["log"]["path"]]
        if (path / "checkpoint-manifest.json").is_file():
            manifest = json.loads((path / "checkpoint-manifest.json").read_text())
            names += ["checkpoint-manifest.json", manifest["checkpoint"]["path"], receipt["results"]["path"]]
    assert len(names) == len(set(names))
    return [{"path": name, "sha256": sha256(base / safe_member(name)), "size_bytes": (base / name).stat().st_size}
            for name in sorted(names)]

def validate_copy(path, marker):
    base = path if marker["directory"] else path.parent
    for entry in marker["files"]:
        target = base / safe_member(entry["path"])
        assert target.is_file() and not target.is_symlink(), f"Missing backup artifact: {target}"
        assert target.stat().st_size == entry["size_bytes"] and sha256(target) == entry["sha256"], f"Backup hash mismatch: {target}"

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    relative = path.relative_to(RUN_BASE).as_posix()
    if relative == "resource-trial":
        relative = "resource-trials/" + sha256(path / "training-receipt.json")[:20]
    destination = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(destination) + ".complete")
    marker = {**workspace_identity, "relative_path": relative, "directory": path.is_dir(), "files": artifact_inventory(path)}
    if marker_path.is_file():
        assert json.loads(marker_path.read_text()) == marker, f"Drive already has different completed evidence: {destination}"
        validate_copy(destination, marker)
        print("Verified existing Drive backup:", destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Only an incomplete copy in this notebook's exact destination is replaced.
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    if path.is_dir():
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path / entry["path"], target)
    else:
        shutil.copyfile(path, destination)
    validate_copy(destination, marker)
    marker_path.write_text(json.dumps(marker, indent=2) + "\n")
    print("Saved verified completed artifacts:", destination, flush=True)

def restore_from_drive(relative):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(source) + ".complete")
    if not marker_path.is_file():
        return None
    marker = json.loads(marker_path.read_text())
    assert all(marker.get(k) == v for k, v in workspace_identity.items()), "Drive completion belongs to different source/protocol pins."
    assert marker["relative_path"] == relative
    validate_copy(source, marker)
    destination = RUN_BASE / safe_member(relative)
    if destination.exists() or destination.is_symlink():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if marker["directory"]:
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / entry["path"], target)
    else:
        assert len(marker["files"]) == 1 and marker["files"][0]["path"] == source.name
        shutil.copyfile(source, destination)
    validate_copy(destination, marker)
    print("Restored completed evidence:", destination, flush=True)
    return destination

print("Workspace:", RUN_BASE)


## 2. Verify the exact experiment

Download all 157 original PNGs and the epoch173 initialization through Git LFS. Replay the **r10 operations against the pinned r9-recovery labels**, checking immutable row identities, source hashes, complete new-series coverage, and the case ledger.

Only eligible training photographs use reviewed supervision. Held photographs and the 31 confirmed old-series treatment exclusions are reported separately; both groups retain exact PR35 labels in both arms. All old-series review files must remain byte-identical to r9. Original 664-front training validation and frozen 666/663 scoring references stay fixed.

Expected status: **ready_for_controlled_incremental_experiment**, with **full_dataset_certified: false**. An inventory, geometry, source-hash, cohort, or label-selection mismatch stops execution. Mechanical checks verify saved decisions; they cannot certify every physical boundary in blurred or occluded pixels.


In [ ]:
lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
if not REPO.exists():
    run(["git", "init", REPO], env=git_env)
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
    run(["git", "fetch", "--depth=1", "origin", SOURCE_REF], cwd=REPO, env=git_env)
    run(["git", "checkout", "--detach", SOURCE_REF], cwd=REPO, env=git_env)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REF
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
assert sha256(REPO / PROTOCOL_PATH) == PROTOCOL_SHA256
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"], cwd=REPO, text=True).splitlines())
assert len(expected_images) == 157 and tracked_images == expected_images
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt", "--exclude="], cwd=REPO)
input_preflight = r"""
import json
from pathlib import Path
import sys
sys.dont_write_bytecode = True
root = Path.cwd()
sys.path.insert(0, str(root / 'scripts'))
from evaluate_pose import file_sha256, preflight_dataset
from verify_incremental_pose_candidate import verify_incremental_candidate
from stage_pose_training import source_state
from rescore_reference_amendment import load_api, verify_amendment
from replay_corner_reference import reference_counts
protocol_path = root / sys.argv[1]
assert file_sha256(protocol_path) == sys.argv[2]
protocol = json.loads(protocol_path.read_text())
candidate = root / protocol['candidate']['path']
inspection = verify_incremental_candidate(root, candidate)
assert inspection['training_readiness'] == 'ready_for_controlled_incremental_experiment'
assert inspection['full_dataset_certified'] is False
assert inspection['retained_photo_labels_identical_between_arms'] is True
assert inspection['unresolved_case_ids'] == protocol['unresolved_case_ids']
assert inspection['retained_legacy_image_ids'] == protocol['retained_legacy_image_ids']
assert inspection['changed_images_vs_control'] == protocol['changed_images_vs_control']
held = set(inspection['retained_legacy_image_ids'])
excluded = set(inspection['excluded_training_image_ids'])
assert inspection['excluded_photo_labels_identical_between_arms'] is True
assert protocol['excluded_training_image_ids'] == sorted(excluded)
records = json.loads((root / 'data/manifest.json').read_text())['records']
old_ids = {record['id'] for record in records if record['id'].startswith('old_')}
new_ids = sorted(record['id'] for record in records if record['id'].startswith('new_'))
assert len(old_ids) == 31 and excluded == old_ids and not held.intersection(excluded)
assert not (held | excluded).intersection(inspection['changed_images_vs_control'])
review_path = root / protocol['source_candidate']['path']
review = json.loads(review_path.read_text())
parent_path = root / review['parent']['path']
assert file_sha256(parent_path) == review['parent']['sha256']
coverage_path = root / review['request_coverage']['path']
assert file_sha256(coverage_path) == review['request_coverage']['sha256']
coverage = json.loads(coverage_path.read_text())
assert len(new_ids) == 126 and coverage['reviewed_image_ids'] == new_ids
assert coverage['visual_review_complete_for_requested_photos'] is True
for record in records:
    if record['id'] in old_ids:
        assert (review_path.parent / record['label']).read_bytes() == (parent_path.parent / record['label']).read_bytes()

assert protocol['evaluation']['current_review_reference'] == protocol['source_candidate']
assert protocol['evaluation']['current_review_reference_is_exploratory'] is True
state = source_state(root, candidate)
initial = preflight_dataset(root, require_weights=True)
assert len(state['records']) == 157
assert state['source']['excluded_training_image_ids'] == sorted(excluded)
for record in records:
    if record['id'] in held | excluded:
        assert state['plans']['reference'][record['label']]['sha256'] == state['plans']['candidate'][record['label']]['sha256']
assert state['source']['initial_checkpoint_sha256'] == initial['model_sha256']
assert state['counts']['reference']['val'] == state['counts']['candidate']['val']
for role, spec in protocol['roles'].items():
    assert spec['arm'] == role
    assert all(state['source'][key] == spec[key] for key in
               ('candidate_manifest', 'candidate_manifest_sha256', 'candidate_id'))
    assert state['counts'][role] == spec['counts']
    assert state['counts'][role]['train']['images'] == 137
    assert state['counts'][role]['train']['empty_images'] == 13
_, api = load_api(root)
assert protocol['evaluation']['diagnostic_reference_required'] is True
references = {}
for role in ('primary', 'diagnostic'):
    reference_path = root / protocol['evaluation'][role + '_reference']
    assert file_sha256(reference_path) == protocol['evaluation'][role + '_reference_sha256']
    reference, gold, *_ = verify_amendment(root, reference_path.parent, initial, api)
    counts, groups = reference_counts(gold)
    assert counts == reference['counts'] == protocol['evaluation'][role + '_counts']
    assert groups == reference['objects_by_locatable_corners']
    references[role] = {'counts': counts, 'objects_by_locatable_corners': groups}
print(json.dumps({'status':inspection['training_readiness'],
                  'full_dataset_certified':False, 'original_images':157,
                  'training_counts':state['counts'],
                  'complete_review_training_counts':protocol['complete_review_counts']['train'],
                  'weights_verified':initial['weights_verified'],
                  'changed_training_photos':len(inspection['changed_images_vs_control']),
                  'held_training_photos':inspection['retained_legacy_image_ids'],
                  'confirmed_old_photos_excluded_from_treatment':inspection['excluded_training_image_ids'],
                  'old_review_files_preserved_from_r9':31,
                  'new_photos_visually_reviewed':126,
                  'remaining_training_cases':len(inspection['unresolved_case_ids']),
                  'remaining_primary_reference_rows':len(inspection['unresolved_reference_rows']),
                  'remaining_historical_reference_rows':len(inspection['diagnostic_unresolved_reference_rows']),
                  'evaluation_references':references}, indent=2))
"""
run([sys.executable, "-B", "-c", input_preflight, PROTOCOL_PATH, PROTOCOL_SHA256], cwd=REPO)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified clean pinned source:", actual_revision)


## 3. Install the pinned trainer

Keep the runtime's matching CUDA Torch and torchvision pair. Install the previously used Ultralytics 8.4.182 under constraints so installation cannot silently replace them. A version conflict needs a compatible environment before continuing.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before installing the trainer."
print("GPU:", torch.cuda.get_device_name(0))
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")), encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__)"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Limits:600 seconds for the trial;21600 seconds for each complete run.")


## 4. Restore completed work and run a resource trial

A saved arm is reused only if its source, protocol, labels, recipe, checkpoint and runtime checks pass. The trial contains four candidate-label batches and never initializes a full run. A failed or incomplete run is preserved for diagnosis. Before retrying, move only its incomplete output folder to an archive location and keep the same `RUN_BASE` and repository paths. Completed arms can then be verified and reused; saved receipts are bound to their workspace paths.


In [ ]:
sys.dont_write_bytecode = True
if str(REPO / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO / "scripts"))
from evaluate_pose import preflight_dataset
from package_gpu_results import collect_arm, check_pair
from verify_incremental_pose_candidate import verify_incremental_candidate

def validate_protocol(root, protocol_path):
    path = root / protocol_path
    document = json.loads(path.read_text())
    receipt = verify_incremental_candidate(root, root / document['candidate']['path'])
    return {'document': document, 'sha256': sha256(path), 'candidate_receipt': receipt}

def collect_incremental_run(preflight, output, role, context):
    bundle = collect_arm(preflight, Path(output).resolve(), role)
    receipt, staging = bundle['receipt'], bundle['staging']
    spec = context['document']['roles'][role]
    assert receipt['arm'] == role
    assert all(receipt[key] == spec[key] for key in ('recipe_id', 'recipe_sha256'))
    assert receipt['runtime']['device'].startswith('cuda:')
    assert receipt['initial_checkpoint_sha256'] == context['document']['initial_checkpoint_sha256']
    assert all(staging['source'][key] == spec[key] for key in
               ('candidate_manifest', 'candidate_manifest_sha256', 'candidate_id'))
    assert staging['source']['training_protocol'] == {'path': PROTOCOL_PATH, 'sha256': context['sha256']}
    assert staging['source'].get('excluded_training_image_ids') == context['document']['excluded_training_image_ids'], 'Completed run omits or changes the confirmed old-photo exclusions.'
    for arm in ROLES:
        assert staging['variants'][arm]['counts'] == context['document']['roles'][arm]['counts']
    return bundle

def check_incremental_pair(bundles, context):
    assert set(bundles) == set(ROLES)
    check_pair(bundles['reference'], bundles['candidate'])
    assert all(bundles[role]['receipt']['arm'] == role for role in ROLES)
from pose_checkpoint import (get_recipe, recipe_identity, training_label_identity, _check_staging, check_training_settings)
from train_pose_comparison import source_identity
CONTEXT = validate_protocol(REPO, PROTOCOL_PATH)
PREFLIGHT = preflight_dataset(REPO, require_weights=False)
assert CONTEXT["sha256"] == PROTOCOL_SHA256
assert CONTEXT["document"]["experiment_id"] == EXPERIMENT_ID
assert set(CONTEXT["document"]["roles"]) == set(ROLES)
for role in ROLES:
    spec = CONTEXT["document"]["roles"][role]
    assert spec["arm"] == role, "Protocol role and actual runner arm must agree."
    assert spec["recipe_id"] == ROLE_RECIPES[role]
    assert spec["recipe_sha256"] == recipe_identity(ROLE_RECIPES[role])["recipe_sha256"] == RECIPE_SHA256
assert get_recipe(ROLE_RECIPES["reference"]) == get_recipe(ROLE_RECIPES["candidate"]), "Both runs must use identical numerical recipes."
for key in ("candidate_manifest", "candidate_manifest_sha256", "candidate_id"):
    assert len({CONTEXT["document"]["roles"][role][key] for role in ROLES}) == 1, "Both arms must use the same incremental candidate identity."


def assert_finished(output, role, *, trial=False):
    assert role in ROLES
    assert not trial or role == "candidate", "Resource trials use candidate labels only."
    output = Path(output)
    if not trial:
        bundle = collect_incremental_run(PREFLIGHT, output, role, CONTEXT)
        assert bundle["receipt"]["arm"] == role, "Completed receipt uses the wrong runner arm."
        assert bundle["receipt"]["source"] == source_identity(REPO), "Completed arm comes from a different implementation."
        return bundle["receipt"]
    receipt = json.loads((output / "training-receipt.json").read_text())
    staging = json.loads((output / "staging-manifest.json").read_text())
    spec = CONTEXT["document"]["roles"][role]
    variants = _check_staging(PREFLIGHT, staging)
    variant = variants[role]
    assert receipt['data_yaml_sha256'] == variant['data_yaml_sha256'], 'Trial dataset YAML identity differs.'
    stage_root = Path(staging['output_root'])
    assert stage_root.is_absolute() and stage_root.name == 'staged-data'
    check_training_settings(receipt['training_settings'], spec['recipe_id'],
                            model=REPO / 'weights/cassette_lower_epoch173.pt',
                            data=stage_root / variant['data_yaml'],
                            device=receipt['training_settings']['device'], project=stage_root.parent)
    assert receipt["status"] == "trial-complete" and receipt["purpose"] == "resource-trial"
    assert spec["arm"] == role == "candidate"
    assert receipt["arm"] == role and receipt["source"] == source_identity(REPO)
    assert all(receipt[k] == spec[k] for k in ("recipe_id", "recipe_sha256"))
    assert all(staging["source"][k] == spec[k] for k in ("candidate_manifest", "candidate_manifest_sha256", "candidate_id"))
    assert receipt["initial_checkpoint_sha256"] == CONTEXT["document"]["initial_checkpoint_sha256"]
    assert receipt["inputs_reverified_after_worker"] is True
    assert receipt["execution"]["exit_code"] == 0 and receipt["execution"]["stop_reason"] is None
    assert receipt["batches_completed"] == 4 and receipt["images_processed"] == 16, "A recoverable trial must contain exactly four full batches."
    samples = receipt["batch_samples"]
    assert len(samples) == 4, "Trial evidence must contain all four observed batches."
    for sample in samples:
        shape = sample["shape"]
        assert len(shape) == 4 and shape[:2] == [4, 3] and min(shape[2:]) > 0 and max(shape[2:]) == 1280, "Trial batches must retain batch 4 and the 1280 long side; rectangular padding is expected."
    assert all(receipt["training_settings"][key] == value for key, value in get_recipe(spec["recipe_id"]).items()), "Recovered trial settings differ from the fixed 30-epoch recipe."
    assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
    log = output / safe_member(receipt["log"]["path"])
    assert log.is_file() and not log.is_symlink() and sha256(log) == receipt["log"]["sha256"], "Resource-trial log hash mismatch."
    assert receipt["runtime"]["device"].startswith("cuda:")
    assert receipt["staging_manifest_sha256"] == sha256(output / "staging-manifest.json")
    assert receipt['training_labels'] == training_label_identity(staging, role)
    assert staging['source']['training_protocol'] == {'path': PROTOCOL_PATH, 'sha256': CONTEXT['sha256']}
    assert staging['source'].get('excluded_training_image_ids') == CONTEXT['document']['excluded_training_image_ids'], 'Resource trial omits or changes the confirmed old-photo exclusions.'
    assert not (output / "checkpoint-manifest.json").exists()
    return receipt


def assert_current_runtime(receipt, current, label):
    expected = receipt["runtime"]
    assert all(expected[k] == v for k, v in current.items() if k != "cudnn_version"), f"Current runtime differs from {label}; select a matching runtime before full training."
    assert expected["backend"]["cudnn_version"] == current["cudnn_version"], f"Current cuDNN runtime differs from {label}; no full arm started."


def train_one(role, *, trial=False):
    assert role in ROLES
    assert not trial or role == "candidate", "Resource trials use candidate labels only."
    recipe_id = CONTEXT["document"]["roles"][role]["recipe_id"]
    folder = "resource-trial" if trial else role
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        try:
            receipt = assert_finished(output, role, trial=trial)
        except Exception as error:
            raise FileExistsError(f"Preserve incomplete or incompatible run {output}. Archive only this incomplete folder, then retry with the same RUN_BASE and repository paths; completed arms can be reused. {error}") from error
        print("Skipping verified completed run:", folder, flush=True)
        save_to_drive(output)
        return output, receipt
    command = [sys.executable, "-B", "scripts/train_pose_comparison.py",
               "--trial" if trial else "--train", "--arm", role, "--recipe", recipe_id,
               "--candidate", CONTEXT["document"]["roles"][role]["candidate_manifest"],
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, role, recipe_id, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}
    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass
    started, last_notice = time.monotonic(), -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}:{elapsed:.0f}s; status={progress.get('status')}; epochs={len(progress.get('completed_epochs', []))}/30; batches={progress.get('batches_completed', 0)}; updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}:{elapsed:.0f}s; verifying and staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed; preserve its receipt/log and archive only the incomplete output folder and retry with the same workspace paths.")
    except BaseException:
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = assert_finished(output, role, trial=trial)
    save_to_drive(output)
    return output, receipt


def recover_trial(current):
    local = RUN_BASE / "resource-trial"
    if local.exists() or local.is_symlink():
        return train_one("candidate", trial=True)
    if DRIVE_ROOT is not None:
        markers = sorted((DRIVE_ROOT / "resource-trials").glob("*.complete"),
                         key=lambda path: path.stat().st_mtime, reverse=True)
        for marker in markers:
            relative = marker.relative_to(DRIVE_ROOT).as_posix().removesuffix(".complete")
            try:
                restored = restore_from_drive(relative)
                receipt = assert_finished(restored, "candidate", trial=True)
                assert_current_runtime(receipt, current, "resource trial")
                for completed in COMPLETED.values():
                    assert receipt["runtime"] == completed["runtime"], "Saved trial differs from a completed arm."
            except (AssertionError, ValueError, OSError, KeyError, TypeError) as error:
                print("Skipped incompatible saved trial:", relative, str(error))
                continue
            print("Reused verified completed trial:", restored)
            return restored, receipt
    return train_one("candidate", trial=True)


COMPLETED = {}
for role in ROLES:
    restore_from_drive(role)
    output = RUN_BASE / role
    if output.exists() or output.is_symlink():
        COMPLETED[role] = assert_finished(output, role)
        print("Recovered eligible completed arm:", role)
TRIAL_RUNTIME = None
if len(COMPLETED) == 2:
    check_incremental_pair({role: collect_incremental_run(PREFLIGHT, RUN_BASE / role, role, CONTEXT) for role in ROLES}, CONTEXT)
    print("Both arms are complete; no resource trial or training will repeat.")
else:
    assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or choose a larger-memory runtime before training."
    run(["nvidia-smi"])
    probe = r"""
import importlib.metadata as m, json, platform, torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'
assert (torch.ones(1, device='cuda:0') * 2).item() == 2
print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,
'ultralytics':m.version('ultralytics'),'torch_cuda':torch.version.cuda,'platform':platform.platform(),
'device':'cuda:0','device_name':torch.cuda.get_device_name(0),'device_capability':list(torch.cuda.get_device_capability(0)),
'packages':{n:m.version(n) for n in ('torchvision','numpy','Pillow','opencv-python','psutil')},
'cudnn_version':torch.backends.cudnn.version()}))
"""
    current = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True, cwd=REPO))
    for role, receipt in COMPLETED.items():
        assert_current_runtime(receipt, current, f"completed {role}")
    TRIAL, trial_receipt = recover_trial(current)
    assert_current_runtime(trial_receipt, current, "resource trial")
    TRIAL_RUNTIME = trial_receipt["runtime"]
    for role, receipt in COMPLETED.items():
        assert receipt["runtime"] == TRIAL_RUNTIME, f"Actual trial runtime differs from completed {role}; no full arm started."
    print("Resource trial passed; it is not a completed model.")


## 5. Train the two missing arms

The two runs use identical numerical settings and the same initialization, runtime and frozen validation split. Only the training supervision changes. This cell runs only after the review gate and resource trial pass.


In [ ]:
REFERENCE_RUN, reference_receipt = train_one("reference")
if TRIAL_RUNTIME is not None:
    assert reference_receipt["runtime"] == TRIAL_RUNTIME, "Reference runtime differs from the trial; do not start the other arm."
CANDIDATE_RUN, candidate_receipt = train_one("candidate")
assert reference_receipt["runtime"] == candidate_receipt["runtime"], "Completed arms have different runtimes; do not report a matched result."
for role, folder in (("reference", REFERENCE_RUN), ("candidate", CANDIDATE_RUN)):
    receipt = assert_finished(folder, role)
    print(role, "runner arm:", receipt["arm"], "final checkpoint:", receipt["checkpoint"]["sha256"])
print("Both matched arms completed. Package the unchanged artifacts next.")


## 6. Package the completed models

Validate both final checkpoints, receipts, logs, staging inventories and their matched settings. The return ZIP records the exact protocol identity through its dataset provenance. Original input images and epoch173 initialization are excluded from the result ZIP.


In [ ]:
from package_gpu_results import package_results, INDEX_NAME, verify_archive
import zipfile
RESULTS_ZIP = RUN_BASE / 'baomai-front-review-r10-results.zip'
restore_from_drive(RESULTS_ZIP.relative_to(RUN_BASE).as_posix())
bundles = {role: collect_incremental_run(PREFLIGHT, RUN_BASE / role, role, CONTEXT) for role in ROLES}
check_incremental_pair(bundles, CONTEXT)
if RESULTS_ZIP.exists():
    with zipfile.ZipFile(RESULTS_ZIP) as archive:
        index_bytes = archive.read(INDEX_NAME)
    index = json.loads(index_bytes)
    assert index['source'] == bundles['reference']['staging']['source']
    assert index['training_source'] == bundles['reference']['receipt']['source']
    assert all(index['arms'][role]['checkpoint_identity'] == bundles[role]['identity'] for role in ROLES)
    rows = sorted((row for bundle in bundles.values() for row in bundle['files']), key=lambda row: row['path'])
    verify_archive(RESULTS_ZIP, index_bytes, rows)
    print('Verified existing result ZIP:', RESULTS_ZIP)
else:
    print(json.dumps(package_results(REPO, REFERENCE_RUN, CANDIDATE_RUN, RESULTS_ZIP), indent=2))
save_to_drive(RESULTS_ZIP)


## 7. Return the ZIP

Download the ZIP or retrieve its verified copy from Drive. Return it for evaluation of both fresh models and retained PR35 against **both pinned development references**, reported separately. Training itself does not establish that the new candidate is better.


In [ ]:
assert RESULTS_ZIP.is_file(), "Complete the packaging cell first."
try:
    from google.colab import files
except ImportError:
    print("Completed results ZIP:", RESULTS_ZIP)
else:
    files.download(str(RESULTS_ZIP))


### What to check after the run

Evaluate **both fresh final-epoch arms and retained PR35 on both fixed references**. The **666-front reference is primary**; the **663-front historical reference is required**. Report them separately. The current r10 development review is exploratory only, with its own hash, counts, and unresolved rows. Equal object counts do not imply equal labels. Trainer validation continues to use the original 664-front labels.

Keep the predeclared box confidence threshold **0.13**, endpoint confidence threshold **0.50**, NMS IoU **0.70**, matching IoU **0.50**, and radii **5 and 10 original-image pixels**. Report scored misses first, then unmatched outputs, duplicates, empty scenes, groups with zero/one/two locatable corners, and corner-pair successes with eligible misses retained in the denominator. Inspect the identities gained and lost by the candidate.

A scored miss means no qualifying one-to-one box match. A retained overlapping prediction can still miss because its IoU falls below the frozen threshold; that differs from an absent prediction. Preserve the fixed scores and inspect original pixels before attributing the failure. The [completed PR76 recovery audit](https://github.com/Siuuuers/dl/tree/848242c6aedc0a521daf0471e1eda514431f5684/results/evaluation/pr76-recovery-20261119) did not support replacing PR35, which remains selected.

A prediction on a reference-missing side is an availability disagreement, not a measurable hidden-coordinate error. Confidence does not prove physical visibility. Each endpoint uses its independent application confidence gate; draw a connecting line only when both pass.

The run does not settle unresolved front/rim or occlusion ownership. Held photographs keep exact PR35 supervision in both arms, as do the separately reported confirmed old-series treatment exclusions. No seed, epoch, threshold, or NMS search is included. Model replacement requires a separate evidence-based decision; reused development scenes do not establish reliability on new camera sessions or robot positioning.
